# Type Hinting & Pydantic — Data Validation

In [ ]:
# Day 36 — Type Hinting & Pydantic
#
# Python is dynamically typed. This means a variable can be an integer,
# then a string, then a list. This is great for quick scripts, but TERRIBLE
# for production systems like MLOps pipelines or APIs.
#
# The Problem:
#   def process_data(data): return data * 2
#   What happens if you pass an integer? (4)
#   What happens if you pass a string? ("datadata")
#   What happens if you pass a list? ([1, 2, 1, 2])
#
# The Solution:
#   1. Type Hinting: Tell developers and IDEs what type is expected.
#   2. Pydantic: Actually ENFORCE the types at runtime and parse data automatically.
#
# Today we'll cover:
#   1. The `typing` module (List, Dict, Optional)
#   2. Pydantic `BaseModel`
#   3. Data Validation with `Field`
#   4. Config Management (Loading Environment Variables)
#
# Install: pip install pydantic pydantic-settings

### 1. Type Hinting Basics

In [ ]:
# Type hints don't STOP the code from running with wrong types.
# They act as documentation for your IDE (like VSCode) to show warnings.

from typing import List, Dict, Optional, Union

# Basic hints
def calculate_total(price: float, tax_rate: float) -> float:
    return price + (price * tax_rate)

# Lists and Dicts
def extract_server_names(servers: List[Dict[str, str]]) -> List[str]:
    return [server['name'] for server in servers]

# Optional (Means it can be the type, OR it can be None)
def get_user_email(user_id: int) -> Optional[str]:
    if user_id == 1:
        return "admin@company.com"
    return None

# Union (Means it can be Type A OR Type B)
def process_id(identifier: Union[int, str]) -> str:
    return f"Processed ID: {identifier}"

print(calculate_total(100.0, 0.2))
print(extract_server_names([{'name': 'web-01'}, {'name': 'db-01'}]))
print(get_user_email(2))

### 2. Introduction to Pydantic

In [ ]:
# Type hints are passive. Pydantic is ACTIVE.
# Pydantic forces the data to match your type hints, or it throws a clear error.
# (This is what FastAPI uses under the hood!).

from pydantic import BaseModel, ValidationError

class ServerConfig(BaseModel):
    name: str
    cpu_cores: int
    ram_gb: float
    is_active: bool = True  # Default value if not provided

# 1. Perfect Data
server1 = ServerConfig(name="web-01", cpu_cores=4, ram_gb=16.0)
print("Server 1:", server1)

# 2. Automatic Type Casting (Pydantic is smart!)
# It converts string "8" to int 8, and int 32 to float 32.0, string "yes" to True.
server2 = ServerConfig(name="db-01", cpu_cores="8", ram_gb=32, is_active="yes")
print("Server 2:", server2)

# 3. Validation Error (Failing gracefully)
try:
    # 'sixteen' cannot be converted to a float!
    bad_server = ServerConfig(name="cache-01", cpu_cores=2, ram_gb="sixteen")
except ValidationError as e:
    print("\nCAUGHT ERROR:\n", e)

### 3. Advanced Validation with Field

In [ ]:
# Sometimes a type isn't enough. 
# You need to restrict the values (e.g. CPU cannot be negative).
# We use Pydantic's `Field` for this.

from pydantic import Field, EmailStr
# Note: pip install email-validator is needed for EmailStr

class MLModelConfig(BaseModel):
    # String constraints
    model_name: str = Field(..., min_length=3, max_length=50, description="The name of the model")
    
    # Numeric constraints (ge = greater or equal, le = less or equal)
    learning_rate: float = Field(0.01, gt=0, lt=1, description="Must be between 0 and 1")
    epochs: int = Field(..., ge=1, le=1000)
    
    # Regex constraints (e.g. must start with 'v' followed by numbers)
    version: str = Field(..., pattern=r'^v[0-9]+\.[0-9]+$')

try:
    config = MLModelConfig(
        model_name="RF", # Too short! min_length is 3
        learning_rate=1.5, # Too high! lt is 1
        epochs=-5, # Too low! ge is 1
        version="1.0" # Doesn't match regex (missing 'v')
    )
except ValidationError as e:
    print("Validation Errors:\n")
    for error in e.errors():
        print(f"Field: {error['loc'][0]} -> {error['msg']}")

### 4. Config Management (Environment Variables)

In [ ]:
# In DevOps/MLOps, you NEVER hardcode secrets (like DB passwords or API keys).
# You load them from Environment Variables.
# Pydantic has `BaseSettings` which automatically reads env variables for you!

import os
from pydantic_settings import BaseSettings

# Let's fake some environment variables in our system
os.environ['DB_HOST'] = 'production.db.azure.com'
os.environ['DB_PORT'] = '5432'
os.environ['API_KEY'] = 'super_secret_key'

class AppSettings(BaseSettings):
    db_host: str
    db_port: int = 3306 # Default fallback if not in env
    api_key: str
    debug_mode: bool = False
    
    class Config:
        # If the env vars had a prefix like APP_DB_HOST, we would set it here:
        # env_prefix = "APP_"
        pass

# Instantiating AppSettings AUTOMATICALLY reads os.environ!
settings = AppSettings()

print(f"Connecting to DB at: {settings.db_host}:{settings.db_port}")
print(f"API Key Length: {len(settings.api_key)}")
print(f"Debug Mode: {settings.debug_mode}")

In [ ]:
# Practice Questions:
#
# 1. Create a Type Hinted function `filter_servers` that takes a list of server dictionaries
#    and a status string, and returns a list of strings (the server names).
#
# 2. Create a Pydantic Model `User` with:
#    - username (min 3 chars)
#    - age (between 18 and 100)
#    - role (must be exactly "admin", "user", or "viewer") Hint: use `Literal` from typing.
#
# 3. Create a Settings class for an MLflow connection that automatically loads
#    MLFLOW_TRACKING_URI and MLFLOW_EXPERIMENT_NAME from the environment.

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: K8s Config Validator
# Write a Pydantic model `KubernetesDeployment` that parses a dictionary
# representing a K8s deployment. It must validate:
# - apiVersion (must be 'apps/v1')
# - replicas (must be between 1 and 100)
# - image (must contain a ':' for the tag, use regex)

# Exercise 2: MLOps Hyperparameter Config Validator
# You are building an ML training script. Use Pydantic to validate the input JSON config:
# - model_type (Literal: "xgboost", "random_forest")
# - n_estimators (int, >0, <1000)
# - learning_rate (Optional float, default 0.1)
# Try passing a bad JSON config and catch the ValidationError gracefully.

# Why this matters for me as a DevOps/MLOps engineer:
# 1. Configuration errors are the #1 cause of production outages. Pydantic catches them instantly.
# 2. FastAPI heavily relies on Pydantic for API requests.
# 3. Parsing complex YAML/JSON configs (like K8s or MLflow setups) into safe Python objects makes automation robust.